# 05. Calibration, Missingness, and Per-SNP Metrics

Calibration adjusts genotype posterior probabilities after HMM inference. It matters because hard calls and downstream association tests depend not only on dosage accuracy, but also on whether the model knows when it is uncertain.

STITCHV2 calibrates posteriors by default. Original STITCH does not expose the same posterior calibration layer, so this is a STITCHV2 advantage rather than a parity requirement.


In [1]:
from pathlib import Path
import os, sys, json, math, shutil, time

REPO = Path('/home/bonnie/Documents/codex/STITCHV2')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
sys.path.insert(0, str(REPO / 'src'))
FIG_DIR = REPO / 'docs' / 'tutorial_deep' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = REPO / 'benchmark_runs' / 'synth_5mb_2k_0p1x'
OUT_DIR = REPO / 'benchmark_runs' / 'tutorial_deep_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
print('repo:', REPO)
print('synthetic data exists:', DATA_DIR.exists())


repo: /home/bonnie/Documents/codex/STITCHV2
synthetic data exists: True


## What Calibration Does

The basic calibration path mixes the raw HMM genotype posterior with a dosage-derived posterior:

```text
calibrated_GP = (1 - blend) * raw_GP + blend * dosage_to_GP(dosage, temperature, depth)
```

- Lower `temperature` makes dosage-derived GP sharper.
- Higher `blend` trusts the dosage-derived posterior more.
- Optional LightGBM calibration can learn correctness probabilities from features such as dosage, depth, support, local context, and posterior confidence.

Calibration does not change the read data. It changes how strongly the model expresses uncertainty and how hard-call/no-call decisions are made.


In [2]:
from stitchv2.calibration import dosage_to_genotype_posterior

dosage_grid = np.linspace(0, 2, 201, dtype=np.float32)[None, :]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, temp in zip(axes, [0.15, 0.35, 0.8]):
    gp = dosage_to_genotype_posterior(dosage_grid, temperature=temp, ploidy=2)[0]
    for g in range(3):
        ax.plot(dosage_grid[0], gp[:, g], label=f'GP{g}')
    ax.set_title(f'temperature={temp}')
    ax.set_xlabel('dosage')
axes[0].set_ylabel('posterior probability')
axes[-1].legend()
fig.tight_layout()
out = FIG_DIR / '05_temperature_gp_curves.png'
fig.savefig(out, dpi=160)
print('wrote', out)


wrote /home/bonnie/Documents/codex/STITCHV2/docs/tutorial_deep/figures/05_temperature_gp_curves.png


![Temperature-to-GP curves](figures/05_temperature_gp_curves.png)

## Benchmark Calibration Results

The repository includes a calibration benchmark with multiple strategies. The cells below load those outputs, check expected columns, and make aggregate and per-SNP violin plots for `R2`, `accuracy`, `F1`, `INFO`, and missing/no-call rate.


In [3]:
cal_dir = REPO / 'benchmark_runs' / 'tutorial_deep_fresh_2026-05-01' / 'calibration'
summary = pd.read_parquet(cal_dir / 'calibration_strategy_summary.parquet')
variant = pd.read_parquet(cal_dir / 'calibration_variant_metrics.parquet')
required = {'strategy','r2','f1','accuracy','balanced_accuracy','call_rate','no_call_rate','info'}
assert required.issubset(variant.columns), sorted(required - set(variant.columns))
print('aggregate calibration strategies:')
display(summary[['strategy','r2','f1','accuracy','balanced_accuracy','call_rate','no_call_rate','info_mean','n_called','n_total']])
print('variant metrics shape:', variant.shape)
display(variant.head())


aggregate calibration strategies:
                   strategy        r2        f1  accuracy  balanced_accuracy  call_rate  no_call_rate  info_mean  n_called  n_total
0               pure_argmax  0.956337  0.966932  0.971719           0.971097   1.000000      0.000000   0.898932   19200.0  19200.0
1   stitch_no_call_balanced  0.955464  0.987505  0.989261           0.987728   0.916667      0.083333   0.905143   17600.0  19200.0
2  lgbmi_onepass_calibrated  0.955464  0.987505  0.989261           0.987728   0.916667      0.083333   0.905143   17600.0  19200.0
3                    stitch -0.722472  0.383463  0.520218           0.453968   0.814063      0.185937   0.769340   15630.0  19200.0
variant metrics shape: (1600, 12)
      strategy  position        r2        f1  accuracy  balanced_accuracy  accuracy_all  call_rate  no_call_rate  pct_assigned_missing  \
0  pure_argmax     11296  0.897590  0.967361  0.958333           0.967361      0.958333        1.0           0.0                   0.0

In [4]:
metrics = ['r2', 'accuracy', 'f1', 'balanced_accuracy', 'info', 'no_call_rate']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for ax, metric in zip(axes, metrics):
    sub = variant[['strategy', metric]].replace([np.inf, -np.inf], np.nan).dropna()
    labels = list(summary['strategy'])
    data = [sub.loc[sub['strategy'] == label, metric].to_numpy() for label in labels]
    parts = ax.violinplot(data, showmeans=True, showextrema=False)
    for body in parts['bodies']:
        body.set_alpha(0.55)
    ax.set_xticks(range(1, len(labels)+1))
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_title(metric)
    ax.grid(axis='y', alpha=0.25)
fig.suptitle('Per-SNP metric distributions by calibration/calling strategy', y=1.02)
fig.tight_layout()
out = FIG_DIR / '05_calibration_per_snp_violins.png'
fig.savefig(out, dpi=160, bbox_inches='tight')
print('wrote', out)


wrote /home/bonnie/Documents/codex/STITCHV2/docs/tutorial_deep/figures/05_calibration_per_snp_violins.png


![Calibration per-SNP violin plots](figures/05_calibration_per_snp_violins.png)


In [5]:
agg_metrics = ['r2','f1','accuracy','balanced_accuracy','info_mean','no_call_rate']
fig, ax = plt.subplots(figsize=(11, 5))
wide = summary.set_index('strategy')[agg_metrics]
wide.plot(kind='bar', ax=ax)
ax.set_title('Aggregate calibration/calling metrics')
ax.set_ylabel('metric value')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
out = FIG_DIR / '05_calibration_aggregate_bar.png'
fig.savefig(out, dpi=160)
print('wrote', out)


wrote /home/bonnie/Documents/codex/STITCHV2/docs/tutorial_deep/figures/05_calibration_aggregate_bar.png


![Calibration aggregate metrics](figures/05_calibration_aggregate_bar.png)

## How To Interpret These Plots

- If `argmax` has high call rate but poor calibration, it may look confident everywhere, including unsupported or ambiguous loci.
- STITCH-style no-call thresholds trade call rate for reliability by assigning missing calls when posterior confidence is low.
- Learned or blended calibration can improve uncertainty estimates and downstream INFO behavior, but it should be validated on held-out truth or pseudo-truth.
- Per-SNP violin plots are important because aggregate metrics can hide a tail of badly behaved variants.
